<a href="https://colab.research.google.com/github/VISHWAJA-028/ML_Multimodality/blob/main/app_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 66.0 MB/s eta 0:00:00


In [2]:
import streamlit
print(streamlit.__version__)

1.64.0


In [3]:
import os
os.environ["PCOS_BUNDLE"] = "/content/drive/MyDrive/PCOS/artifacts/pcos_multimodal_bundle.pkl"

!streamlit run app.py --server.port 8501 &>/content/streamlit.log &

In [4]:
import os
for f in ["app.py", "pcos_predictor.py"]:
    print(f, "->", os.path.exists(f"/content/drive/MyDrive/PCOS/{f}"))

app.py -> False
pcos_predictor.py -> False


In [5]:
import glob, json, sys, os

from google.colab import drive
drive.mount('/content/drive')

hits = glob.glob("/content/drive/MyDrive/**/pcos_predictor_py.ipynb", recursive=True)
print("found:", hits)
nb = json.load(open(hits[0]))

# cells 2, 3, 4 = imports, constants, functions + class (no Drive mount, no tests)
code = "\n\n".join("".join(nb["cells"][i]["source"]).strip("\n") for i in (2, 3, 4))
assert "class PCOSPredictor" in code, "Wrong cells picked - tell me and I'll adjust"

open("/content/pcos_predictor.py", "w").write(code + "\n")
os.chdir("/content")
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

from pcos_predictor import PCOSPredictor
print("pcos_predictor ready")

Mounted at /content/drive
found: ['/content/drive/MyDrive/PCOS/artifacts/pcos_predictor_py.ipynb']
pcos_predictor ready


In [6]:
%%writefile /content/app.py
import os

import streamlit as st

from pcos_predictor import DISCLAIMER, PCOSPredictor

BUNDLE_PATH = os.environ.get("PCOS_BUNDLE", "pcos_bundle.pkl")

st.set_page_config(page_title="PCOS Screening Assistant", page_icon="🩺", layout="wide")


@st.cache_resource(show_spinner="Loading models (first run takes a minute)...")
def get_predictor():
    return PCOSPredictor.load(BUNDLE_PATH)


def yn(choice):
    return {"Yes": "Y", "No": "N"}.get(choice)


def result_card(title, res):
    p = float(res["prob_pcos"])
    st.subheader(title)
    c1, c2 = st.columns(2)
    c1.metric("Prediction", res["label"])
    c2.metric("PCOS probability", f"{p * 100:.1f}%")
    st.progress(min(max(p, 0.0), 1.0))
    if 0.4 <= p <= 0.6:
        st.warning("Borderline result - clinical review is strongly advised.")
    for w in res.get("warnings", []):
        st.info(w)


try:
    predictor = get_predictor()
except Exception as e:
    st.error(f"Could not load '{BUNDLE_PATH}': {e}")
    st.stop()

st.title("🩺 PCOS Screening Assistant")
st.caption(DISCLAIMER)

with st.sidebar:
    st.header("About")
    st.write("Two models: a tabular model (clinical features) and an "
             "ultrasound CNN. Use either one, or both.")
    m = predictor.meta.get("image_test_metrics")
    if m:
        st.write(f"**Ultrasound model:** {predictor.meta.get('image_backbone')}")
        st.write({k: round(v, 3) for k, v in m.items() if v is not None})
    tab_weight = st.slider("Weight of clinical data when using both", 0.0, 1.0, 0.5, 0.05)

mode = st.radio("Input type", ["Clinical data", "Ultrasound image", "Both"], horizontal=True)
features, image_bytes = None, None

if mode in ("Clinical data", "Both"):
    st.header("Clinical data")
    a, b, c = st.columns(3)
    age = a.number_input("Age (yrs)", 10, 60, value=None)
    weight = b.number_input("Weight (Kg)", 25.0, 200.0, value=None)
    height = c.number_input("Height (Cm)", 120.0, 210.0, value=None)
    if weight and height:
        st.caption(f"Calculated BMI: {weight / (height / 100) ** 2:.1f}")

    a, b, c, d = st.columns(4)
    cycle = a.selectbox("Menstrual cycle", ["Not sure", "Regular", "Irregular"])
    hair = b.selectbox("Excess hair growth", ["Not sure", "No", "Yes"])
    gain = c.selectbox("Recent weight gain", ["Not sure", "No", "Yes"])
    skin = d.selectbox("Skin darkening", ["Not sure", "No", "Yes"])

    a, b, c = st.columns(3)
    fol_l = a.number_input("Follicle No. (Left ovary)", 0, 40, value=None)
    fol_r = b.number_input("Follicle No. (Right ovary)", 0, 40, value=None)
    amh = c.number_input("AMH (ng/mL)", 0.0, 60.0, value=None)

    with st.expander("Optional extra values"):
        a, b, c = st.columns(3)
        fsh_lh = a.number_input("FSH/LH ratio", 0.0, 50.0, value=None)
        fsz_l = b.number_input("Avg. follicle size L (mm)", 0.0, 40.0, value=None)
        fsz_r = c.number_input("Avg. follicle size R (mm)", 0.0, 40.0, value=None)
        blood = st.selectbox("Blood group", ["Unknown", "A+", "A-", "B+", "B-", "O+", "O-", "AB+", "AB-"])

    raw = {
        "Age (yrs)": age, "Weight (Kg)": weight, "Height(Cm)": height,
        "Cycle(R/I)": {"Regular": "R", "Irregular": "I"}.get(cycle),
        "hair growth(Y/N)": yn(hair), "Weight gain(Y/N)": yn(gain),
        "Skin darkening (Y/N)": yn(skin),
        "Follicle No. (L)": fol_l, "Follicle No. (R)": fol_r, "AMH(ng/mL)": amh,
        "FSH/LH": fsh_lh, "Avg. F size (L) (mm)": fsz_l, "Avg. F size (R) (mm)": fsz_r,
        "Blood Group": None if blood == "Unknown" else blood,
    }
    features = {k: v for k, v in raw.items() if v is not None}

if mode in ("Ultrasound image", "Both"):
    st.header("Ultrasound image")
    up = st.file_uploader("Upload an ovarian ultrasound (JPG/PNG)", type=["jpg", "jpeg", "png"])
    if up is not None:
        image_bytes = up.getvalue()
        st.image(image_bytes, caption="Uploaded image", width=320)

st.divider()
if st.button("Predict", type="primary"):
    if mode in ("Clinical data", "Both") and not features:
        st.error("Please enter at least a few clinical values.")
        st.stop()
    if mode in ("Ultrasound image", "Both") and image_bytes is None:
        st.error("Please upload an ultrasound image.")
        st.stop()
    try:
        with st.spinner("Predicting..."):
            out = predictor.predict(
                features=features if mode != "Ultrasound image" else None,
                image=image_bytes if mode != "Clinical data" else None,
                tab_weight=tab_weight,
            )
    except Exception as e:
        st.error(f"Prediction failed: {e}")
        st.stop()

    cols = st.columns(2 if out["mode"] == "both" else 1)
    i = 0
    if "structured" in out:
        with cols[i]:
            result_card("Clinical-data model", out["structured"])
        i += 1
    if "image" in out:
        with cols[i]:
            result_card("Ultrasound model", out["image"])
    if "combined" in out:
        st.divider()
        result_card("Combined result", out["combined"])
        if not out["combined"]["agree"]:
            st.warning("The two models disagree - refer for clinical review.")
        st.caption(out["combined"]["note"])
    st.caption(out["disclaimer"])

Writing /content/app.py


In [7]:
!pip install -q streamlit

In [2]:
import os
os.chdir("/content")
from google.colab import drive
drive.mount('/content/drive')

print("app.py:", os.path.exists("app.py"))
print("pcos_predictor.py:", os.path.exists("pcos_predictor.py"))
print("bundle:", os.path.exists("/content/drive/MyDrive/PCOS/artifacts/pcos_multimodal_bundle.pkl"))
!pip install -q streamlit

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
app.py: True
pcos_predictor.py: True
bundle: False


In [3]:
import glob, os
print("Files in artifacts:")
for f in sorted(os.listdir("/content/drive/MyDrive/PCOS/artifacts")):
    p = os.path.join("/content/drive/MyDrive/PCOS/artifacts", f)
    print(f"  {f}  ->  {os.path.getsize(p)/1e6:.1f} MB")

print("\nAnywhere in Drive:")
for p in glob.glob("/content/drive/MyDrive/**/*bundle*", recursive=True):
    print(f"  {p}  ->  {os.path.getsize(p)/1e6:.1f} MB")

Files in artifacts:
  PCOS_best_image_model.keras  ->  212.0 MB
  PCOS_image_model_info.json  ->  0.0 MB
  model_comparison.csv  ->  0.0 MB
  pcos_bundle.pkl  ->  213.1 MB
  pcos_predictor_py.ipynb  ->  0.1 MB
  structured_bundle.joblib  ->  1.1 MB
  structured_columns.joblib  ->  0.0 MB
  structured_imputer.joblib  ->  0.0 MB
  structured_model.joblib  ->  1.1 MB
  structured_model_comparison.csv  ->  0.0 MB
  structured_model_info.json  ->  0.0 MB
  structured_scaler.joblib  ->  0.0 MB

Anywhere in Drive:
  /content/drive/MyDrive/PCOS/artifacts/structured_bundle.joblib  ->  1.1 MB
  /content/drive/MyDrive/PCOS/artifacts/pcos_bundle.pkl  ->  213.1 MB


In [4]:
import os, sys, subprocess, time
os.chdir("/content")

env = dict(os.environ,
           PCOS_BUNDLE="/content/drive/MyDrive/PCOS/artifacts/pcos_bundle.pkl")
log = open("/content/streamlit.log", "w")
st = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port", "8501", "--server.headless", "true"],
    stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True)

time.sleep(20)
print("still running:", st.poll() is None)
r = subprocess.run(["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}",
                    "http://localhost:8501"], capture_output=True, text=True)
print("http status:", r.stdout)

still running: True
http status: 200


In [5]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

import re
proc = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8501"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        print("Open this link:", m.group(0))
        break

Open this link: https://dpi-determines-unavailable-cleared.trycloudflare.com
